# Many qubits: GHZ and the exponential wall
**Quantum Computing · Week 4 (Entanglement) · Day 4**

Lesson page: https://sara-aminpour.com/tutorials/qc/w4d4.html

In [ ]:
# On Google Colab, this installs qiskit (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2

GHZ states on many qubits, and the exponential cost of writing a quantum state down.

In [ ]:
import time
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.primitives import StatevectorSampler

def ghz(n):
    """H on qubit 0, then a chain of CNOTs: (|00...0> + |11...1>)/sqrt(2)."""
    qc = QuantumCircuit(n)
    qc.h(0)
    for q in range(n - 1):
        qc.cx(q, q + 1)
    return qc

print(ghz(3).draw("text"))
print({str(k): round(float(v), 3) for k, v in Statevector(ghz(3)).probabilities_dict().items()})

## One measurement decides them all
Ten qubits, 1000 shots: only two bitstrings ever appear.

In [ ]:
qc = ghz(10)
qc.measure_all()
counts = StatevectorSampler(seed=2).run([qc], shots=1000).result()[0].data.meas.get_counts()
print(counts)
print("gates:", ghz(10).size(), "  depth:", ghz(10).depth())

## The exponential wall
$n$ qubits need $2^n$ complex amplitudes, 16 bytes each.

In [ ]:
def human(nbytes):
    for unit in ["bytes", "KB", "MB", "GB", "TB", "PB", "EB", "ZB", "YB"]:
        if nbytes < 1024:
            return f"{nbytes:,.0f} {unit}"
        nbytes /= 1024
    return f"{nbytes:,.0f} more than YB"

for n in [10, 20, 30, 40, 50, 60]:
    print(f"{n:2d} qubits: {2 ** n:>22,d} amplitudes = {human(16 * 2 ** n):>12}")

## Feel it: simulation time doubles with every qubit

In [ ]:
import matplotlib.pyplot as plt

ns, secs = list(range(12, 23)), []
for n in ns:
    t0 = time.perf_counter()
    Statevector(ghz(n))
    secs.append(time.perf_counter() - t0)
plt.figure(figsize=(6, 3.2))
plt.semilogy(ns, secs, "o-", color="#7b5cfa")
plt.xlabel("number of qubits")
plt.ylabel("seconds (log scale)")
plt.title("Each qubit roughly doubles the work")
plt.show()

## Your turn: a faster GHZ
The chain needs $n-1$ CNOTs one after another: depth $n$. But a qubit that is already "in" can help:
after `cx(0, 1)`, qubits 0 and 1 can each copy into a new qubit **at the same time**. Build GHZ on 8 qubits
with depth 4 (one H, then three layers of CNOTs), and check that it's the same state as the chain.

In [ ]:
fast = QuantumCircuit(8)
fast.h(0)
# TODO: layer 1: 0->1;  layer 2: 0->2, 1->3;  layer 3: 0->4, 1->5, 2->6, 3->7
print("depth:", fast.depth(), "  same state as the chain?", Statevector(fast).equiv(Statevector(ghz(8))))

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
fast = QuantumCircuit(8)
fast.h(0)
fast.cx(0, 1)
for c, t in [(0, 2), (1, 3)]:
    fast.cx(c, t)
for c, t in [(0, 4), (1, 5), (2, 6), (3, 7)]:
    fast.cx(c, t)
print(fast.draw("text"))
print("depth:", fast.depth(), "  same state as the chain?", Statevector(fast).equiv(Statevector(ghz(8))))
print("The number of qubits in the GHZ doubles every layer, so n qubits need only about log2(n) layers.")